In [2]:
import soundfile as sf
import io
import numpy as np
import base64
import requests
import subprocess
import librosa
from IPython.display import Audio

In [3]:
SAMPLE_RATE = 16000
# URL = "https://api.sangjeong.com:8080/whisper/stt_duration"
URL = "http://localhost:8000/asr/stt_duration"

In [4]:
def compress_to_opus(bytes: bytes) -> bytes:
  process = subprocess.Popen(
    ["ffmpeg", "-i", "pipe:0", "-c:a", "libopus", "-f", "opus", "pipe:1"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE # subprocess.DEVNULL 하면 속도 조금 더 빨라짐
  )

  out, err = process.communicate(input=bytes)
  return out, err 

In [5]:
def np_to_wav(audio: np.ndarray, sample_rate:int) -> bytes:
  buffer = io.BytesIO()
  sf.write(buffer, audio, sample_rate, format='wav')
  return buffer.getvalue()

In [6]:
audio, sr = librosa.load(".data/news_with_english.mp3", sr=SAMPLE_RATE)

In [7]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=4000, scale=400))
  rand_len = np.clip(rand_len, 3500, 4500)
  end = min(pos + rand_len, total_samples)

  chunk = audio[pos:end]
  segments.append(chunk)
  pos = end

In [8]:
idx = 0

In [9]:
audio = segments[idx]
idx += 1
Audio(audio, rate=SAMPLE_RATE)

In [29]:
output = []
for segment in segments:
  audio = np_to_wav(segment, SAMPLE_RATE)
  # audio, _ = compress_to_opus(audio)
  audio_b64 = base64.b64encode(audio).decode('utf-8')
  params = {
    "group": "4", 
    "user": "1", 
    "audio": audio_b64
  }

  res = requests.get(URL, params=params)
  # print(res)
  d = res.json()
  print(d)
  output = [ *output, *d['completed'] ]

{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': [{'order': 0, 'lang': ['ko'], 'text': 'BBC 생방송 인터뷰도 중에 자녀', 'words': [{'start': 2880.0, 'end': 11520.0, 'text': ' BBC', 'lang': 'ko'}, {'start': 11520.0, 'end': 22400.0, 'text': ' 생방송', 'lang': 'ko'}, {'start': 22400.0, 'end': 30720.0, 'text': ' 인터뷰도', 'lang': 'ko'}, {'start': 30720.0, 'end': 35840.0, 'text': ' 중에', 'lang': 'ko'}, {'start': 35840.0, 'end': 47040.0, 'text': ' 자녀', 'lang': 'ko'}]}]}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'comple

In [30]:
for s in output:
  print(s["text"])
  print(f"\t start:{s['words'][0]['start']}, end:{s['words'][-1]['end']}")

BBC 생방송 인터뷰 중에 자녀 난입 사건으로 스타가 된 미국인 교수 가족이 오늘 카메라 앞에 섰습니다.
	 start:2880.0, end:123963.0
유튜브 스타가 된 4살짜리 딸은 이번엔 사탕을 입에 물고 등장했습니다.
	 start:131643.0, end:211361.0
배영진 기자입니다.
	 start:216491.0, end:234721.0
BBC 인터뷰 도중 딸과 아들의 등장으로 일약 스타가 된 로버트 켈리 부산대 교수.
	 start:279251.0, end:371589.0
유튜브 영상 조회비가,600만 건이 넘는 켈리 교수 가족은 세계적인 유명 됐습니다.
	 start:377191.0, end:477829.0
이렇게 언론의 관심이 커지자 켈리 교수 가족이 기자회견에 나섰습니다.
	 start:485242.0, end:561242.0
춤을 췄던 첫째 딸 메리아는 사탕을 물었고 둘째 둘째 아들 존은 엄마 품에 안긴 모습이었습니다. 
	 start:570682.0, end:751630.0
thought it was a disaster. 
	 start:762738.0, end:778738.0
I immediately called or texted or emailed the BBC. 
	 start:786738.0, end:836786.0
I communicated with the BBC immediately afterwards, and I apologized to them. 
	 start:838386.0, end:891836.0
I said that if they never us back or never asked me to be on television again, I would understand.
	 start:892476.0, end:971760.0
귀여운 춤으로 화제가 된 딸에 대한 질문도 잇따랐습니다.
	 start:981672.0, end:1033058.0
당시 BBC와 인터뷰한 내용은 한국의 대통령 탄핵 사건이었습니다.
	 st